In [7]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EsmForMaskedLM, 
    EsmTokenizer
)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, matthews_corrcoef
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
import seaborn as sns
import time
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader



SEED = 42
os.environ["WANDB_MODE"] = "disabled"

# Set seeds for reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) 


In [8]:
def extract_embeddings(model_path, sequences, batch_size=256):
    tokenizer = EsmTokenizer.from_pretrained(model_path)
    model = EsmForMaskedLM.from_pretrained(model_path)
    model.eval()
    model = model.cuda()

    all_embeddings = []

    for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting features"):
        batch = sequences[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # Extract [CLS] embeddings
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Use last layer [CLS] token
            embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        
        all_embeddings.append(embeddings)
    
    return np.vstack(all_embeddings)

In [9]:
dataset = Dataset.from_csv("experiments/dataset/AMP-dataset/AMP.csv").rename_column('label', 'labels')
split = dataset.train_test_split(test_size=0.2, seed=SEED)

test_dataset = Dataset.from_csv("experiments/dataset/test-dataset/abp.indtest1.csv").rename_column('label', 'labels')

In [10]:
train_sequences = split["train"]["seq_name"]
train_labels    = split["train"]["labels"]
test_sequences  = test_dataset["seq_name"]
test_labels     = test_dataset["labels"]

# Extract features using YOUR TRAINED STUDENT MODEL
print("Extracting features from distilled ESM-2...")
X_train = extract_embeddings("facebook/esm2_t6_8M_UR50D", train_sequences)
X_test = extract_embeddings("facebook/esm2_t6_8M_UR50D", test_sequences)

y_train = np.array(train_labels)
y_test = np.array(test_labels)

print(f"Feature shape: {X_train.shape}")
print(f"Train samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")

Extracting features from distilled ESM-2...


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extracting features: 100%|██████████| 59/59 [00:05<00:00, 10.76it/s]


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extracting features: 100%|██████████| 4/4 [00:00<00:00,  9.28it/s]


Feature shape: (15045, 320)
Train samples: 15045
Test samples: 1000


In [11]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Train and evaluate a model"""
    print(f"\n{'='*70}")
    print(f"Training {model_name}...")
    print(f"{'='*70}")
    
    # Train
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Predict
    start_time = time.time()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    inference_time = time.time() - start_time
    
    # Metrics
    results = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_proba),
        'mcc': matthews_corrcoef(y_test, y_pred),
        'train_time': train_time,
        'inference_time': inference_time,
        'inference_speed': len(y_test) / inference_time,
    }
    
    print(f"Accuracy:  {results['accuracy']:.4f}")
    print(f"F1-Score:  {results['f1']:.4f}")
    print(f"AUC:       {results['auc']:.4f}")
    print(f"MCC:       {results['mcc']:.4f}")
    print(f"Train time: {train_time:.2f}s")
    print(f"Inference: {results['inference_speed']:.0f} samples/sec")
    
    return results

### XGBoost

In [12]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=42,
    eval_metric='logloss'
)

xgb_results = evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")


Training XGBoost...
Accuracy:  0.9710
F1-Score:  0.9712
AUC:       0.9962
MCC:       0.9421
Train time: 54.28s
Inference: 5096 samples/sec


### Random forest

In [13]:
rf_model = RandomForestClassifier(
    n_estimators=100,      # Good balance
    max_depth=20,          # Prevent overfitting
    min_samples_split=5,
    n_jobs=-1,             # Use all CPU cores
    random_state=42,
    verbose=0
)

rf_results = evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest")


Training Random Forest...


Accuracy:  0.9610
F1-Score:  0.9612
AUC:       0.9947
MCC:       0.9220
Train time: 6.54s
Inference: 12839 samples/sec


### ANN

In [14]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[256, 128, 64]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 2))  # Binary classification
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

In [15]:
def train_nn(X_train, y_train, X_test, y_test, epochs=20):
    """Train simple NN"""
    
    # Prepare data
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.LongTensor(y_test)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
    
    # Model
    model = SimpleANN(input_dim=X_train.shape[1])
    model = model.cuda()
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Training
    print(f"\n{'='*70}")
    print("Training Simple NN...")
    print(f"{'='*70}")
    
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.cuda(), batch_y.cuda()
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if (epoch + 1) % 5 == 0:
            avg_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    
    train_time = time.time() - start_time
    
    # Evaluation
    model.eval()
    with torch.no_grad():
        X_test_cuda = X_test_t.cuda()
        logits = model(X_test_cuda)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
    
    # Inference time
    start_time = time.time()
    with torch.no_grad():
        _ = model(X_test_cuda)
    inference_time = time.time() - start_time
    
    # Metrics
    results = {
        'model': 'Simple ANN',
        'accuracy': accuracy_score(y_test, preds),
        'f1': f1_score(y_test, preds),
        'auc': roc_auc_score(y_test, probs),
        'mcc': matthews_corrcoef(y_test, preds),
        'train_time': train_time,
        'inference_time': inference_time,
        'inference_speed': len(y_test) / inference_time,
    }
    
    print(f"\nResults:")
    print(f"Accuracy:  {results['accuracy']:.4f}")
    print(f"F1-Score:  {results['f1']:.4f}")
    print(f"AUC:       {results['auc']:.4f}")
    print(f"MCC:       {results['mcc']:.4f}")
    
    return results, model

ann_results, ann_model = train_nn(X_train, y_train, X_test, y_test)


Training Simple NN...
Epoch 5/20, Loss: 0.1813
Epoch 10/20, Loss: 0.1690
Epoch 15/20, Loss: 0.1491
Epoch 20/20, Loss: 0.1461

Results:
Accuracy:  0.9640
F1-Score:  0.9647
AUC:       0.9960
MCC:       0.9287


### CNN

In [16]:
class SimpleCNN(nn.Module):
    def __init__(self, vocab_size=33, embedding_dim=128):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Multiple filter sizes (like Kim 2014 for text)
        self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embedding_dim, 128, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(embedding_dim, 128, kernel_size=7, padding=3)
        
        self.pool = nn.AdaptiveMaxPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(128 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    
    def forward(self, x):
        # x: [batch_size, seq_len]
        x = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        x = x.transpose(1, 2)  # [batch_size, embedding_dim, seq_len]
        
        # Apply convolutions
        x1 = torch.relu(self.conv1(x))
        x2 = torch.relu(self.conv2(x))
        x3 = torch.relu(self.conv3(x))
        
        # Pool
        x1 = self.pool(x1).squeeze(2)
        x2 = self.pool(x2).squeeze(2)
        x3 = self.pool(x3).squeeze(2)
        
        # Concatenate
        x = torch.cat([x1, x2, x3], dim=1)
        
        # Classify
        x = self.fc(x)
        return x

# Training similar to ANN above...
# (Use tokenized sequences, not embeddings)
cnn_results, cnn_model = train_nn(X_train, y_train, X_test, y_test)


Training Simple NN...
Epoch 5/20, Loss: 0.1759
Epoch 10/20, Loss: 0.1596
Epoch 15/20, Loss: 0.1534
Epoch 20/20, Loss: 0.1446

Results:
Accuracy:  0.9750
F1-Score:  0.9750
AUC:       0.9958
MCC:       0.9500
